# Conservative 2D regrid — regions (grid → country/state polygons)

A common analysis question is: *given a gridded variable, what's its
area-weighted mean over each of these regions?* — countries, states,
watersheds, ocean basins, exclusive economic zones. This is the canonical
[xagg](https://github.com/ks905383/xagg)-style workflow and a natural fit
for **conservative regridding**: each region's output value is the
area-weighted average of the source cells it overlaps, which is exactly
what you want for fluxes and intensive quantities (precipitation,
temperature, mass-balance budgets) where bilinear or nearest-neighbor
interpolation would bias the total.

Because the targets are arbitrary polygons rather than a grid, the fast
`.regrid.conservative` accessor doesn't apply. We use
`ConservativeRegridder.from_polygons`, which takes a 1D array of shapely
polygons as source and target.

**In this notebook.**

1. Aggregate xarray's air-temperature tutorial dataset (NMC reanalysis,
   ~daily 2.5° lat/lon over North America) onto US states built by
   dissolving county boundaries from `geodatasets`.
2. Visualize the per-state means against the source grid.
3. Verify conservation directly from the internal weight matrix.
4. Save the regridder, reload it, and reuse it on JJA vs. DJF subsets to
   get per-state seasonal swings — showing the weight matrix is reusable
   across any source field on the same grid.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
import geodatasets

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder, polygons_from_coords

## Source — NCEP reanalysis surface air temperature

`xr.tutorial.open_dataset("air_temperature")` is a small (~3 MB) NMC
reanalysis subset: 4×daily surface air temperature over North America on
a 2.5° lat/lon grid. We reduce it to a long-term mean, convert from
Kelvin to °C, and align with the state polygons' CRS:

- Native longitudes are on `[0, 360)`; states are on `[-180, 180)`, so
  we wrap.
- Native latitudes are descending; we sort to ascending so xarray's
  `pcolormesh` doesn't flip the image.

In [ ]:
ds = xr.tutorial.open_dataset("air_temperature")
air = (ds["air"].mean("time") - 273.15).sortby("lat")
air = air.assign_coords(lon=(((air.lon + 180) % 360) - 180)).sortby("lon")
air.attrs["units"] = "degC"
air.name = "mean_air_temperature"
air

## Regions — US states from `geodatasets`

`geodatasets.get_path("geoda.ncovr")` returns a GeoPackage of the 49
contiguous-US counties (+ DC). We **dissolve** on `STATE_NAME` —
geopandas' equivalent of a `groupby` for geometries — to combine each
state's counties into one (Multi)Polygon. The result is exactly what
`ConservativeRegridder.from_polygons` wants: one shapely (Multi)Polygon
per region. The air-temperature grid doesn't cover Alaska or Hawaii,
which is why ncovr (contiguous-US only) is a natural fit.

In [ ]:
counties = gpd.read_file(geodatasets.get_path("geoda.ncovr"))
states = (
    counties.dissolve(by="STATE_NAME", aggfunc="first")
    .reset_index()[["STATE_NAME", "geometry"]]
    .sort_values("STATE_NAME")
    .reset_index(drop=True)
)
print(f"{len(states)} states/DC, CRS={states.crs}, "
      f"bounds={states.total_bounds.round(1).tolist()}")
states.head(3)

## Build the regridder and apply

`from_polygons` takes flat 1D arrays of source and target polygons:

- **Source polygons** come from the 1D grid coords via
  `polygons_from_coords`, which builds a rectangle per cell from the
  coordinate midpoints (so a 25×53 grid → 1325 source polygons).
- **Target polygons** are just the states' `geometry` column.

Source data has to be flattened to a single `src_cell` dimension to match
the flat polygon array. Constructing the regridder is the expensive step
— it computes the area of every source/target polygon intersection.
Once built, `rgr.regrid(...)` is a sparse matrix-vector product against
the precomputed weights and is essentially instant.

In [ ]:
src_polys = polygons_from_coords(air.lon.values, air.lat.values)
tgt_polys = states.geometry.to_numpy()

rgr = ConservativeRegridder.from_polygons(
    source_polygons=src_polys,
    target_polygons=tgt_polys,
    source_dim="src_cell",
    target_dim="state",
    target_coords=xr.Dataset(coords={"state": states.STATE_NAME.values}),
)

src_flat = xr.DataArray(air.values.ravel(), dims=("src_cell",))
state_mean = rgr.regrid(src_flat)
state_mean.attrs["units"] = "degC"
state_mean.to_series().sort_values().round(2)

## Map + ranked bar chart

States filled by area-weighted mean temperature, with the source grid
shown underneath for reference. Both panels share the same `vmin`/`vmax`
so a state's color on the bar chart matches its color on the map. A
sanity check: the warm/cool gradient should track latitude (Florida and
the Gulf states warmest, the Upper Midwest and Northeast coolest).

In [ ]:
fig, (ax_map, ax_bar) = plt.subplots(
    1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.6, 1]},
)

air.plot(ax=ax_map, cmap="coolwarm", alpha=0.5,
         cbar_kwargs={"shrink": 0.65, "label": "grid mean T [°C]"})

vmin, vmax = float(air.min()), float(air.max())
states_plot = states.assign(mean_T=state_mean.values)
states_plot.plot(
    column="mean_T", cmap="coolwarm", ax=ax_map,
    edgecolor="black", linewidth=0.4,
    vmin=vmin, vmax=vmax,
)
ax_map.set_xlim(-128, -65); ax_map.set_ylim(22, 52)
ax_map.set_title("Annual mean surface T — states vs. source grid")
ax_map.set_xlabel("longitude"); ax_map.set_ylabel("latitude")

ordered = state_mean.to_series().sort_values()
colors = plt.cm.coolwarm((ordered.values - vmin) / (vmax - vmin))
ax_bar.barh(ordered.index, ordered.values, color=colors)
ax_bar.set_xlabel("area-weighted mean T [°C]")
ax_bar.tick_params(axis="y", labelsize=7)
ax_bar.grid(axis="x", alpha=0.3)
plt.tight_layout()

## Conservation check

The internal area matrix `A[i, j] = area(state_i ∩ src_cell_j)` lets us
verify conservation directly: integrating the source field weighted by
source-cell coverage should equal integrating the regridded field
weighted by target-cell area. Equality to machine precision is the
defining property of *conservative* regridding.

In [ ]:
A = rgr.areas                                # (n_states, n_src)
tgt_area = rgr.target_areas
src_cover = rgr.source_coverage_areas

direct = float((air.values.ravel() * src_cover).sum())
via_regrid = float((state_mean.values * tgt_area).sum())
print(f"direct   A·s            : {direct:.6f}")
print(f"Σ state_mean · a_state  : {via_regrid:.6f}")
print(f"relative error          : {abs(direct - via_regrid) / abs(direct):.2e}")

## Reuse: persist the regridder, apply to summer vs. winter

The weight matrix depends only on the source/target geometry, not on the
data. Save once with `to_netcdf`, reload with `from_netcdf`, and apply
to any field on the same source grid — no need to rebuild the (expensive)
polygon intersection. Below we use one saved regridder on JJA and DJF
subsets to compute per-state seasonal amplitude. Continental interior
states show the largest swing; Florida and California — moderated by
ocean and latitude — show the smallest.

In [ ]:
import tempfile
from pathlib import Path
path = Path(tempfile.gettempdir()) / "states_regridder.nc"
rgr.to_netcdf(path)
print(f"wrote {path.name}  ({path.stat().st_size / 1024:.1f} KB)")

rgr2 = ConservativeRegridder.from_netcdf(path)

def seasonal_mean(months):
    sub = ds["air"].sel(time=ds["time.month"].isin(months)).mean("time") - 273.15
    sub = sub.sortby("lat")
    sub = sub.assign_coords(lon=(((sub.lon + 180) % 360) - 180)).sortby("lon")
    flat = xr.DataArray(sub.values.ravel(), dims=("src_cell",))
    return rgr2.regrid(flat)

summer = seasonal_mean([6, 7, 8])
winter = seasonal_mean([12, 1, 2])
amplitude = (summer - winter).to_series().rename("JJA − DJF [°C]").round(1)
print("largest seasonal swing:")
print(amplitude.sort_values(ascending=False).head(5))
print("\nsmallest seasonal swing (maritime / subtropical):")
print(amplitude.sort_values().head(5))